In [ ]:
from skmultilearn.dataset import load_from_arff
import numpy as np

# ---- Train ----
X_train, y_train = load_from_arff(
    "plantgo-train.arff",
    label_count=12,        # PlantGO has 12 labels
    label_location="end",
    load_sparse=True
)

# ---- Test ----
X_test, y_test = load_from_arff(
    "PlantGO-test.arff",
    label_count=12,
    label_location="end",
    load_sparse=True
)

# Convert sparse → dense (if your SVM needs NumPy)
X_train = X_train.toarray().astype(np.float32)
X_test = X_test.toarray().astype(np.float32)

y_train = y_train.toarray().astype(np.int32)
y_test = y_test.toarray().astype(np.int32)

print("Train X:", X_train.shape)
print("Train y:", y_train.shape)
print("Test X:", X_test.shape)
print("Test y:", y_test.shape)

# Save
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

##### SVM Multi Label Algorithm

**Training**
Input: X (n×d), Y (n×L) where Y_ij ∈ {0,1}
For each label j in 1..L:
    model_j = BinarySVM()
    model_j.fit(X, Y[:, j])
Store all models {model_j}

**Prediction**
Input: X
For each label j:
    score_j = model_j.decision_function(X)
    pred_j = 1 if score_j >= threshold_j else 0
Return predictions matrix (n×L)

In [ ]:
class KernelSVM:
    def __init__(self, C=1.0, kernel='rbf', sigma=0.1, alpha=1, c=0, degree=2, max_kernel_samples=2500, random_state=42):
        self.C = C
        self.max_kernel_samples = max_kernel_samples
        self.random_state = random_state
        self.kernel_name = kernel

        if kernel == 'linear':
            self.kernel = self.linear_kernel
        elif kernel == 'rbf':
            self.kernel = self.rbf_kernel
            self.sigma = sigma
        elif kernel == 'poly':
            self.kernel = self.poly_kernel
            self.alpha = alpha
            self.c = c
            self.degree = degree
        elif kernel == 'sigmoid':
            self.kernel = self.sigmoid_kernel
            self.alpha = alpha
            self.c = c
        else:
            raise ValueError("Unsupported kernel type. Choose from 'linear', 'rbf', 'poly', or 'sigmoid'.")

        self.X = None
        self.y = None
        self.lmbda = None
        self.b = 0.0
        self.w = None

    def linear_kernel(self, X, Z):
        return X @ Z.T

    def rbf_kernel(self, X, Z):
        X2 = np.sum(X**2, axis=1)[:, None]
        Z2 = np.sum(Z**2, axis=1)[None, :]
        return np.exp(-self.sigma * (X2 + Z2 - 2 * X @ Z.T))

    def poly_kernel(self, X, Z):
        return (self.alpha * X @ Z.T + self.c) ** self.degree

    def sigmoid_kernel(self, X, Z):
        return np.tanh(self.alpha * X @ Z.T + self.c)

    def fit(self, X, y, lr=1e-3, epochs=200):
        X = np.asarray(X, dtype=np.float32)
        y = np.where(np.asarray(y) == 0, -1.0, 1.0).astype(np.float32)

        n = X.shape[0]

        if self.kernel_name == 'linear':
            self.w = np.zeros(X.shape[1], dtype=np.float32)
            self.b = 0.0
            reg = 1.0 / max(self.C, 1e-12)

            for _ in range(epochs):
                margin = y * (X @ self.w + self.b)
                mask = margin < 1.0
                if np.any(mask):
                    grad_w = reg * self.w - np.mean(y[mask, None] * X[mask], axis=0)
                    grad_b = -np.mean(y[mask])
                else:
                    grad_w = reg * self.w
                    grad_b = 0.0
                self.w -= lr * grad_w
                self.b -= lr * grad_b
            return self

        if self.max_kernel_samples is not None and n > self.max_kernel_samples:
            rng = np.random.default_rng(self.random_state)
            pos = np.where(y == 1)[0]
            neg = np.where(y == -1)[0]
            n_sub = self.max_kernel_samples
            n_pos = min(len(pos), n_sub // 2)
            n_neg = min(len(neg), n_sub - n_pos)
            if n_pos + n_neg < n_sub:
                extra = n_sub - (n_pos + n_neg)
                n_pos = min(len(pos), n_pos + extra)
                n_neg = min(len(neg), n_sub - n_pos)

            sub_idx = np.concatenate([
                rng.choice(pos, size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=int),
                rng.choice(neg, size=n_neg, replace=False) if n_neg > 0 else np.array([], dtype=int),
            ])
            rng.shuffle(sub_idx)
            X = X[sub_idx]
            y = y[sub_idx]
            n = X.shape[0]

        self.X = X
        self.y = y
        self.lmbda = np.zeros(n, dtype=np.float32)
        self.b = 0.0

        ones = np.ones(n, dtype=np.float32)
        y_outer = np.outer(y, y)
        K = self.kernel(X, X)
        Q = y_outer * K
        yy = y @ y

        for _ in range(epochs):
            gradient = ones - Q @ self.lmbda
            self.lmbda += lr * gradient

            for _proj in range(2):
                self.lmbda -= y * (y @ self.lmbda) / max(yy, 1e-12)
                self.lmbda = np.clip(self.lmbda, 0.0, self.C)

            idx = np.where((self.lmbda > 1e-8) & (self.lmbda < self.C - 1e-8))[0]
            if len(idx) > 0:
                b_i = y[idx] - (self.lmbda * y) @ K[:, idx]
                self.b = float(np.mean(b_i))

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=np.float32)
        if self.kernel_name == 'linear' and self.w is not None:
            return X @ self.w + self.b

        K = self.kernel(self.X, X)
        return (self.lmbda * self.y) @ K + self.b

    def predict(self, X, threshold=0.0):
        return (self.decision_function(X) >= threshold).astype(np.int32)


class BinaryRelevanceSVM:
    """
    Multi-label wrapper: one KernelSVM per label column.

    threshold_strategy:
      - "zero": fixed threshold 0 for each label
      - "prevalence": per-label threshold matching train label prevalence
    """
    def __init__(self, base_params: dict, fit_params: dict = None, threshold_strategy="prevalence"):
        self.base_params = dict(base_params)
        self.fit_params = dict(fit_params) if fit_params is not None else {}
        self.threshold_strategy = threshold_strategy
        self.models_ = []
        self.thresholds_ = None
        self.n_labels_ = None

    def fit(self, X, Y):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.int32)
        if Y.ndim != 2:
            raise ValueError("Y must be a 2D array of shape (n_samples, n_labels).")

        n_samples = X.shape[0]
        self.n_labels_ = Y.shape[1]
        self.models_ = []
        self.thresholds_ = np.zeros(self.n_labels_, dtype=np.float32)

        for j in range(self.n_labels_):
            y_bin = Y[:, j].astype(np.int32)
            model = KernelSVM(**self.base_params)
            model.fit(X, y_bin, **self.fit_params)
            self.models_.append(model)

            if self.threshold_strategy == "zero":
                self.thresholds_[j] = 0.0
            elif self.threshold_strategy == "prevalence":
                scores = model.decision_function(X)
                k = int(y_bin.sum())
                if k <= 0:
                    self.thresholds_[j] = np.inf
                elif k >= n_samples:
                    self.thresholds_[j] = -np.inf
                else:
                    self.thresholds_[j] = np.partition(scores, n_samples - k)[n_samples - k]
            else:
                raise ValueError("threshold_strategy must be 'zero' or 'prevalence'.")

        return self

    def decision_function(self, X):
        scores = [m.decision_function(X) for m in self.models_]
        return np.vstack(scores).T

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= self.thresholds_[None, :]).astype(np.int32)



##### Evaluation Metrics

1. Hamming Loss: It meansures the fration of label decisions that are wrong (averaged over all samples × labels)
It is a good indicator because it evaluates each label independently (robust when exact match is rare).

2. Micro-averaged Precision / Recall / F1: It aggregates TP/FP/FN across all labels, and emphasizes performance on frequent labels and overall correctness.

3. Macro-averaged F1
Compute F1 per label, then average → treats each label equally.
Best when you care about rare labels too.

4. Subset Accuracy (Exact Match)
Strict: counts a sample correct only if all labels match.
Useful but often low; include as a “hard” metric to show exact multi-label correctness.

In [ ]:
def multilabel_confusion(Y_true, Y_pred):
    # Y_* shape: (n_samples, n_labels) with {0,1}
    tp = np.sum((Y_true == 1) & (Y_pred == 1))
    fp = np.sum((Y_true == 0) & (Y_pred == 1))
    fn = np.sum((Y_true == 1) & (Y_pred == 0))
    tn = np.sum((Y_true == 0) & (Y_pred == 0))
    return tp, fp, fn, tn

def micro_precision_recall_f1(Y_true, Y_pred, eps=1e-12):
    tp, fp, fn, _ = multilabel_confusion(Y_true, Y_pred)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    return prec, rec, f1

def macro_f1(Y_true, Y_pred, eps=1e-12):
    # average F1 across labels
    L = Y_true.shape[1]
    f1s = []
    for l in range(L):
        yt, yp = Y_true[:, l], Y_pred[:, l]
        tp = np.sum((yt==1) & (yp==1))
        fp = np.sum((yt==0) & (yp==1))
        fn = np.sum((yt==1) & (yp==0))
        prec = tp/(tp+fp+eps)
        rec  = tp/(tp+fn+eps)
        f1   = 2*prec*rec/(prec+rec+eps)
        f1s.append(f1)
    return float(np.mean(f1s))

def hamming_loss(Y_true, Y_pred):
    return float(np.mean(Y_true != Y_pred))

def subset_accuracy(Y_true, Y_pred):
    return float(np.mean(np.all(Y_true == Y_pred, axis=1)))

##### Implement and Results


In [ ]:
from sklearn.preprocessing import normalize


def train_test_report(X_train, Y_train, X_test, Y_test, svm_params, fit_params=None, threshold_strategy="prevalence"):
    # L2 normalization stabilizes the hinge optimization on high-dimensional sparse features.
    X_train = normalize(np.asarray(X_train, dtype=np.float32), norm="l2")
    X_test = normalize(np.asarray(X_test, dtype=np.float32), norm="l2")

    model = BinaryRelevanceSVM(
        base_params=svm_params,
        fit_params=fit_params or {},
        threshold_strategy=threshold_strategy,
    )
    model.fit(X_train, Y_train)

    Y_pred_train = model.predict(X_train)
    Y_pred_test = model.predict(X_test)

    def pack_metrics(Yt, Yp):
        p, r, f1 = micro_precision_recall_f1(Yt, Yp)
        return {
            "HammingLoss": hamming_loss(Yt, Yp),
            "SubsetAcc": subset_accuracy(Yt, Yp),
            "MicroP": p,
            "MicroR": r,
            "MicroF1": f1,
            "MacroF1": macro_f1(Yt, Yp),
        }

    train_m = pack_metrics(Y_train, Y_pred_train)
    test_m = pack_metrics(Y_test, Y_pred_test)
    return train_m, test_m


svm_params = {
    "C": 0.1,
    "kernel": "linear",
    "sigma": 0.5,
    "alpha": 1.0,
    "c": 0.0,
    "degree": 3,
    "max_kernel_samples": None,
    "random_state": 42,
}
fit_params = {"lr": 1e-2, "epochs": 500}

train_m, test_m = train_test_report(
    np.load("X_train.npy"), np.load("y_train.npy"),
    np.load("X_test.npy"), np.load("y_test.npy"),
    svm_params=svm_params,
    fit_params=fit_params,
    threshold_strategy="prevalence",
)

print("Training Metrics:")
for k, v in train_m.items():
    print(f"  {k}: {v}")

print("\nTesting Metrics:")
for k, v in test_m.items():
    print(f"  {k}: {v}")



##### SVM Multi Class Algorithm

##### Dataset - MNIST: Modified National Institute of Standards and Technology database
It contains: 70,000 grayscale images; Image size: 28 * 28 pixels; 10 classes (digits 0-9); 60,000 training images; 10,000 test images.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

mnist = fetch_openml('mnist_784', version=1)
X = mnist.data
y = mnist.target.astype(int)

print("MNIST X:", X.shape)
print("MNIST y:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


##### Pseudocode

Input: X (n×d), y (n,), classes = unique(y)
For each class k in classes:
    yk = 1 if y==k else 0
    model_k = BinarySVM()
    model_k.fit(X, yk)
Store all models {model_k}

Input: X
For each class k:
    score_k = model_k.decision_function(X)   # real-valued margins
Return argmax_k score_k for each sample

In [ ]:
class OneVsRestSVM:
    """
    Multi-class wrapper using One-vs-Rest.
    Uses decision_function scores; predicts argmax score.
    """
    def __init__(self, base_params: dict, fit_params: dict = None):
        self.base_params = dict(base_params)
        self.fit_params = dict(fit_params) if fit_params is not None else {}
        self.classes_ = None
        self.models_ = {}

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.models_ = {}

        for cls in self.classes_:
            y_bin = (y == cls).astype(int)
            model = KernelSVM(**self.base_params)
            model.fit(X, y_bin, **self.fit_params)
            self.models_[cls] = model
        return self

    def decision_function(self, X):
        # shape: (n_samples, n_classes)
        scores = []
        for cls in self.classes_:
            scores.append(self.models_[cls].decision_function(X))
        return np.vstack(scores).T

    def predict(self, X):
        scores = self.decision_function(X)
        idx = np.argmax(scores, axis=1)
        return self.classes_[idx]

In [ ]:
def accuracy(y_true, y_pred):
    return float((y_true == y_pred).mean())

def macro_f1(y_true, y_pred, n_classes=10):
    f1s = []
    for c in range(n_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        f1   = 2 * prec * rec / (prec + rec + 1e-12)
        f1s.append(f1)
    return float(np.mean(f1s))

svm_params = {"lr": 0.05, "lambda_param": 1e-4, "n_epochs": 15, "batch_size": 512, "seed": 42}
model = MultiClassSVM(n_classes=10, svm_params=svm_params)
model.fit(np.asanyarray(X_train), np.asanyarray(y_train))

y_pred_train = model.predict(np.asanyarray(X_train))
y_pred_test = model.predict(np.asanyarray(X_test))

print("Train Accuracy:", accuracy(y_train, y_pred_train))
print("Train Macro-F1:", macro_f1(y_train, y_pred_train, n_classes=10))
print("Test  Accuracy:", accuracy(y_test, y_pred_test))
print("Test  Macro-F1:", macro_f1(y_test, y_pred_test, n_classes=10))

**Metrics justification**

MNIST is multi-class with roughly balanced classes.

Accuracy works well when classes are balanced. 

Macro-F1 is also used to evaluate. It treats each digit equally, and reveals if the model is failing on a specific digit.


##### Observation and Analyze

Accuracy and macro-f1 results are very similar between train and test dataset. It indicates good generalization and minimal overfitting. 